# Build the ATT&CK catalog

Builds the catalog that PRISMA loads, `frontend/public/tactics_and_techniques_by_domain.json`, from MITRE's ATT&CK files (STIX) and the extra DeTT&CT data sources in `dettect_files/`. Run all cells (*Kernel → Restart Kernel and Run All Cells*); the catalog is written only when the checks pass.

The first run downloads MITRE's files for `ATTACK_VERSION` (about 50 MB) from [MITRE's ATT&CK STIX data](https://github.com/mitre-attack/attack-stix-data) into `mitre_attack_files/`, and checks them against MITRE's official fingerprints. They are not part of the repository. MITRE's licence for ATT&CK, which also covers the catalog, is in `frontend/public/LICENCE-ATTACK.txt`, next to the catalog.

**New ATT&CK version:** set `ATTACK_VERSION` below and run all cells. If `mitreattack-python` does not know the version yet, update it first (in `requirements.txt`). Also check whether DeTT&CT's data source file has changed (see the DeTT&CT cell), and afterwards refresh the template's technique list (see `templates/`).

In [ ]:
from mitreattack.release_info import STIX21
from mitreattack.stix20 import MitreAttackData
from collections import defaultdict
from pathlib import Path
from urllib.request import urlopen
import hashlib
import json
import os
import tempfile

In [ ]:
# The ATT&CK version, and the files (relative to this notebook's folder, tools/).
ATTACK_VERSION = '17.1'
mitre_enterprise_file_path = f"mitre_attack_files/enterprise-attack-{ATTACK_VERSION}.json"
mitre_ics_file_path = f"mitre_attack_files/ics-attack-{ATTACK_VERSION}.json"
mitre_mobile_file_path = f"mitre_attack_files/mobile-attack-{ATTACK_VERSION}.json"
dettect_file_path = 'dettect_files/dettect_data_sources.json'
catalog_file_path = '../frontend/public/tactics_and_techniques_by_domain.json'

In [ ]:
# Download MITRE's ATT&CK files for this version when they are not here yet, and check every file against MITRE's
# official fingerprint (which the mitreattack-python library knows for each release).
os.makedirs('mitre_attack_files', exist_ok=True)
for domain, path in [('enterprise', mitre_enterprise_file_path), ('ics', mitre_ics_file_path), ('mobile', mitre_mobile_file_path)]:
    expected_hash = STIX21[domain].get(ATTACK_VERSION)
    if expected_hash is None:
        raise ValueError(f"mitreattack-python does not know ATT&CK {ATTACK_VERSION} yet. Update it first (see requirements.txt).")
    if not os.path.exists(path):
        url = f"https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/{domain}-attack/{domain}-attack-{ATTACK_VERSION}.json"
        print(f"Downloading {url}")
        with urlopen(url) as response, tempfile.NamedTemporaryFile('wb', dir='mitre_attack_files', delete=False) as temporary:
            temporary.write(response.read())
        os.replace(temporary.name, path)
    if hashlib.sha256(Path(path).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"{path} is not MITRE's ATT&CK {ATTACK_VERSION} file: its fingerprint is different. Delete it and run this cell again.")
    print(f"{path}: checked")

In [ ]:
# Pre-process information about dettect-specific MITRE data components (which are called data sources in DeTT&CT).
# The file is sourced from: https://github.com/rabobank-cdc/DeTTECT/blob/master/data/dettect_data_sources.json
with open(dettect_file_path) as json_data:
    dettect_data = json.load(json_data)

# Get all unique data components.
dettect_data_components = set()
for x in dettect_data:
    dettect_data_components.update(x['dettect_data_sources'])

# Create a dict mapping technique ids to names on Enterprise domain.
enterprise = MitreAttackData(mitre_enterprise_file_path)
techniques = enterprise.get_techniques(remove_revoked_deprecated=True)
techniques_id_name_dict = {x['external_references'][0]['external_id']: x['name'] for x in techniques}

# Create a dict for all dettect data components and their techniques (sorted, so the output order is the same on every run).
dettect_data_components_dict = {}
for component in sorted(dettect_data_components):
    dettect_data_components_dict[component] = []

# Fill the dettect data components dict.
for x in dettect_data:
    for data_component in x['dettect_data_sources']:
        dettect_data_components_dict[data_component].append({'id': x['technique_id'], 'name': techniques_id_name_dict[x['technique_id']]})

# Create and fill a dict with all dettect techniques and their data components.
dettect_all_techniques_dict = {}
for x in dettect_data:
    dettect_all_techniques_dict[x['technique_id']] = x['dettect_data_sources']

In [ ]:
def get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique_stix_id):

    # get data components detecting technique
    datacomponents_detects_technique = mitre_attack_data.get_datacomponents_detecting_technique(technique_stix_id)

    # Get all data sources and data components.
    data_sources = []
    data_components = []
    for d in datacomponents_detects_technique:
        datacomponent = d["object"]
        datasource = mitre_attack_data.get_object_by_stix_id(datacomponent.x_mitre_data_source_ref)
        data_sources.append(datasource.name)
        data_components.append(datacomponent.name)

    # Remove duplicates in the data sources (sorted, so the output order is the same on every run).
    data_sources = sorted(set(data_sources))
    
    # Remove duplicates in the data components.
    data_components = list(dict.fromkeys(data_components))
    

    return data_sources, data_components

In [ ]:
def process_mitre_attack_json(json_file_path):
    """Process the ATT&CK data from a single ATT&CK json file."""
    
    # Initialize the MitreAttackData class
    mitre_attack_data = MitreAttackData(json_file_path)
    
    # Collect, per parent technique, the groups and software that use the technique or one of its sub-techniques,
    # because using a sub-technique implies using its parent technique.
    domain_techniques = mitre_attack_data.get_techniques(remove_revoked_deprecated=True)
    parent_technique_names = {x['external_references'][0]['external_id']: x['name'] for x in domain_techniques if not x.get('x_mitre_is_subtechnique')}
    rolled_up_groups = defaultdict(set)
    rolled_up_software = defaultdict(set)
    for technique in domain_techniques:
        parent_id = technique['external_references'][0]['external_id'].split('.')[0]
        rolled_up_groups[parent_id].update(x['object']['name'] for x in mitre_attack_data.get_groups_using_technique(technique.id))
        rolled_up_software[parent_id].update(x['object']['name'] for x in mitre_attack_data.get_software_using_technique(technique.id))

    # Fetch all tactics
    tactics = mitre_attack_data.get_tactics_by_matrix()
    
    # Prepare the structure for the JSON
    tactics_entries = []

    # Create a list of all platforms (will have many duplicates)
    all_platforms = []
    
    # We assume that our ATT&CK JSON files contain a single domain, which we extract here
    if 'Enterprise ATT&CK' in list(tactics.keys()):
        domain_str = 'Enterprise ATT&CK'
        domain = 'enterprise-attack'
    # We ignore the 'Network-Based Effects' key that appears in the tactics.keys() for the Mobile domain
    if 'Mobile ATT&CK' in list(tactics.keys()):
        domain_str = 'Mobile ATT&CK'
        domain = 'mobile-attack'
    if 'ATT&CK for ICS' in list(tactics.keys()):
        domain_str = 'ATT&CK for ICS'
        domain = 'ics-attack'
    


    #### Iterate over all tactics
    for tactic in tactics[domain_str]:
        # Fetch all techniques for the current tactic
        techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], domain, remove_revoked_deprecated=True)

        # Prepare the techniques list with sub-techniques nested under their parent techniques
        techniques_dict = {}
        for technique in techniques:
            
            platforms = technique.get('x_mitre_platforms')
            if platforms != None:
                all_platforms.extend(platforms)
            data_sources, data_components = get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique.id)
            groups = mitre_attack_data.get_groups_using_technique(technique.id)
            groups = [x['object']['name'] for x in groups]
            software = mitre_attack_data.get_software_using_technique(technique.id)
            software = [x['object']['name'] for x in software]

            # Parent techniques also count the groups and software that use one of their sub-techniques.
            if not technique.get('x_mitre_is_subtechnique'):
                groups = sorted(rolled_up_groups[technique['external_references'][0]['external_id']])
                software = sorted(rolled_up_software[technique['external_references'][0]['external_id']])
            occurrence_groups = len(groups)
            occurrence_software = len(software)
            
            technique_entry = {
                "name": technique['name'],
                "external_id": technique['external_references'][0]['external_id'],
                "platforms": platforms,
                'groups': groups,
                'occurrence_groups': occurrence_groups,
                'software': software,
                'occurrence_software': occurrence_software,
                'occurrence_total': occurrence_groups + occurrence_software,
                'data_sources': data_sources,
                'data_components': data_components,
                "visibility": False,  # Set default visibility to False, can be modified as needed.
                "visibility_ratio": 0,
            }

            # Add dettect-specific data sources information to all techniques and sub-techniques, for Enterprise domain.
            if domain == 'enterprise-attack':
                if technique_entry['external_id'] in dettect_all_techniques_dict:
                    technique_entry['data_components'].extend(dettect_all_techniques_dict[technique_entry['external_id']])
                    # Remove duplicates while preserving order
                    technique_entry['data_components'] = list(dict.fromkeys(technique_entry['data_components']))

            # Nest sub-techniques under their parent techniques correctly, and add parent techniques to the main techniques_dict.
            if technique.get('x_mitre_is_subtechnique'):
                parent_id = technique['external_references'][0]['external_id'].split('.')[0]
                if parent_id in techniques_dict:
                    if 'sub_techniques' not in techniques_dict[parent_id]:
                        techniques_dict[parent_id]['sub_techniques'] = []
                    techniques_dict[parent_id]['sub_techniques'].append(technique_entry)
                else:
                    techniques_dict[parent_id] = {
                        "sub_techniques": [technique_entry]
                    }
            else:
                # Keep any sub-techniques that were collected before their parent technique was seen.
                placeholder = techniques_dict.get(technique_entry['external_id'], {})
                if 'sub_techniques' in placeholder:
                    technique_entry['sub_techniques'] = placeholder['sub_techniques']
                techniques_dict[technique_entry['external_id']] = technique_entry
    
        # Sort the techniques alphabetically by name
        techniques_list = sorted([value for value in techniques_dict.values() if 'name' in value], key=lambda x: x['name'])
    
        # Sort the sub-techniques alphabetically by name
        for technique in techniques_list:
            if 'sub_techniques' in technique:
                technique['sub_techniques'] = sorted(technique['sub_techniques'], key=lambda x: x['name'])

        # Compute (sub)technique counts for the tactic.
        technique_count = len(techniques_list)
        subtechnique_count = sum([len(x['sub_techniques']) for x in techniques_list if 'sub_techniques' in x])
        all_technique_count = technique_count + subtechnique_count

        # Prepare the tactic entry
        tactic_entry = {
            "name": tactic['name'],
            "external_id": tactic['external_references'][0]['external_id'],
            "techniques": techniques_list,
            "technique_count": technique_count,
            "subtechnique_count": subtechnique_count,
            "all_technique_count": all_technique_count,
        }
        
        tactics_entries.append(tactic_entry)

    
    #### Pre-process the tactics_entries, so we can compute an order for the occurrence.
    total_techniques_count = 0
    for tactic_entry in tactics_entries:
        total_techniques_count += len(tactic_entry['techniques'])

    # Put all techniques in a single list.
    all_techniques = []
    for tactic_entry in tactics_entries:
        all_techniques += tactic_entry['techniques']

    # Assign each technique a percentile rank for its occurrence: the share of the other (unique) techniques that occur
    # less often. Techniques with the same occurrence share a rank. The rank is always between 0 and 1, however large the
    # occurrence counts become, so the Low/Medium/High/Very High filter keeps meaning all/top 75%/top 50%/top 25%.
    def assign_normalized_frequencies(occurrence_key):
        unique_techniques = {technique['external_id']: technique for technique in all_techniques}.values()
        occurrences = [technique[occurrence_key] for technique in unique_techniques]
        for technique in all_techniques:
            lower_occurrences = sum(1 for x in occurrences if x < technique[occurrence_key])
            technique[f'{occurrence_key}_order_normalized'] = lower_occurrences / max(len(occurrences) - 1, 1)

    # Apply the function for each occurrence key.
    assign_normalized_frequencies('occurrence_total')
    assign_normalized_frequencies('occurrence_groups')
    assign_normalized_frequencies('occurrence_software')
    
    
    ### Create platforms entries
    platforms_entries = []
    
    # Remove duplicates in all_platforms list (sorted, so the output order is the same on every run)
    all_platforms = sorted(set(all_platforms))

    # Loop over all platforms
    for platform in all_platforms:
        techniques_by_platform = mitre_attack_data.get_techniques_by_platform(platform, remove_revoked_deprecated=True)
        all_techniques = [x.name for x in techniques_by_platform]
        techniques = [x.name  for x in techniques_by_platform if x.x_mitre_is_subtechnique == False]
        subtechniques = [x.name  for x in techniques_by_platform if x.x_mitre_is_subtechnique == True]


        # Prepare the platform entry
        platform_entry = {
            "name": platform,
            "active_in_filter": True,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        platforms_entries.append(platform_entry)

    # Sort platform entries by technique_count
    platforms_entries = sorted(platforms_entries, key=lambda item: item['all_technique_count'], reverse=True)


    
    #### Create groups entries
    groups_entries = []

    # Get all groups
    groups = mitre_attack_data.get_groups(remove_revoked_deprecated=True)

    # Loop over all groups
    for group in groups:
        techniques_used_by_group = mitre_attack_data.get_techniques_used_by_group(group.id)
        all_techniques = [x['object'].name for x in techniques_used_by_group]
        techniques = [x['object'].name for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == True]

        # Using a sub-technique implies using its parent technique.
        implied_parents = [parent_technique_names[x['object'].external_references[0].external_id.split('.')[0]] for x in techniques_used_by_group if x['object'].x_mitre_is_subtechnique == True]
        techniques = list(dict.fromkeys(techniques + implied_parents))
        all_techniques = list(dict.fromkeys(all_techniques + implied_parents))

        # Prepare group entry
        group_entry = {
            "name": group.name,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        groups_entries.append(group_entry)

    # Sort group entries by technique_count.
    groups_entries = sorted(groups_entries, key=lambda item: item['all_technique_count'], reverse=True)


    
    #### Create software entries
    software_entries = []

    # Get all softwares
    softwares = mitre_attack_data.get_software(remove_revoked_deprecated=True)

    # Loop over all softwares
    for software in softwares:
        techniques_used_by_software = mitre_attack_data.get_techniques_used_by_software(software.id)
        all_techniques = [x['object'].name for x in techniques_used_by_software]
        techniques = [x['object'].name for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == True]

        # Using a sub-technique implies using its parent technique.
        implied_parents = [parent_technique_names[x['object'].external_references[0].external_id.split('.')[0]] for x in techniques_used_by_software if x['object'].x_mitre_is_subtechnique == True]
        techniques = list(dict.fromkeys(techniques + implied_parents))
        all_techniques = list(dict.fromkeys(all_techniques + implied_parents))

        # Prepare software entry
        software_entry = {
            "name": software.name,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
        }

        software_entries.append(software_entry)

    # Sort software entries by technique_count
    software_entries = sorted(software_entries, key=lambda item: item['all_technique_count'], reverse=True)
    

    
    #### Create datasources entries
    datasources_entries = []

    # Get all data sources
    datasources = mitre_attack_data.get_datasources(remove_revoked_deprecated=True)

    # Loop over all datasources
    for datasource in datasources:

        # Prepare datasource entry
        datasource_entry = {
            'name': datasource['name'],
            'active_in_filter': True,
        }
    
        datasources_entries.append(datasource_entry)


    
    #### Create datacomponent entries
    datacomponents_entries = []

    # Get all data components
    datacomponents = mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)

    # Loop over all datacomponens
    for datacomponent in datacomponents:

        # Get all techniques detected by each datacomponent
        techniques_detected_by_datacomponent = mitre_attack_data.get_techniques_detected_by_datacomponent(datacomponent['id'])
        all_techniques = [x['object'].name for x in techniques_detected_by_datacomponent]
        techniques = [x['object'].name for x in techniques_detected_by_datacomponent if x['object'].x_mitre_is_subtechnique == False]
        subtechniques = [x['object'].name for x in techniques_detected_by_datacomponent if x['object'].x_mitre_is_subtechnique == True]

        # Prepare datacomponent entry
        datacomponent_entry = {
            "name": datacomponent['name'],
            "active_in_filter": True,
            "techniques": techniques,
            "subtechniques": subtechniques,
            "all_techniques": all_techniques,
            "technique_count": len(techniques),
            "subtechnique_count": len(subtechniques),
            "all_technique_count": len(all_techniques),
            "visibility": False,
            "quality": {
                "device_completeness": None,
                "data_field_completeness": None,
                "timeliness": None,
                "consistency": None,
                "retention": None,
            }
        }
    
        datacomponents_entries.append(datacomponent_entry)


    # Add dettect specific data components, for Enterprise domain.
    if domain == 'enterprise-attack':
      # Track which data component names we've already added from MITRE ATT&CK to avoid duplicates
      existing_component_names = {entry['name'] for entry in datacomponents_entries}
    
      for key, val in dettect_data_components_dict.items():
          # Skip if this data component already exists (from MITRE ATT&CK)
          if key in existing_component_names:
              continue
    
          all_techniques = [x['name'] for x in val]
          techniques = [x['name'] for x in val if not '.' in x['id']]
          subtechniques = [x['name'] for x in val if '.' in x['id']]
    
          # Prepare datacomponent entry
          datacomponent_entry = {
              "name": key,
              "active_in_filter": True,
              "techniques": techniques,
              "subtechniques": subtechniques,
              "all_techniques": all_techniques,
              "technique_count": len(techniques),
              "subtechnique_count": len(subtechniques),
              "all_technique_count": len(all_techniques),
              "visibility": False,  # Set default visibility to False, like the MITRE ATT&CK data components.
              "quality": {
                  "device_completeness": None,
                  "data_field_completeness": None,
                  "timeliness": None,
                  "consistency": None,
                  "retention": None,
              }
          }
    
          datacomponents_entries.append(datacomponent_entry)
    
    
    # Sort data components entries by technique_count.
    datacomponents_entries = sorted(datacomponents_entries, key=lambda item: item['all_technique_count'], reverse=True)

    
    #### Combine all data for the domain into a single entry.
    domain_entry = {
        "tactics": tactics_entries,
        "platforms": platforms_entries,
        "data_sources": datasources_entries,
        "data_components": datacomponents_entries,
        "groups": groups_entries,
        "softwares": software_entries,
    }

    return domain_entry

In [ ]:
# The 'meta' block of the catalog: the ATT&CK version, and fingerprints of the source files, so you can tell what a
# catalog was built from.

def file_hash(path):
    return 'sha256:' + hashlib.sha256(Path(path).read_bytes()).hexdigest()

def attack_version_of(path):
    """The ATT&CK version that one of MITRE's files states (in its x-mitre-collection object)."""
    with open(path) as json_file:
        return next(x['x_mitre_version'] for x in json.load(json_file)['objects'] if x['type'] == 'x-mitre-collection')

source_files = [mitre_enterprise_file_path, mitre_ics_file_path, mitre_mobile_file_path]
versions = sorted({attack_version_of(path) for path in source_files})
if versions != [ATTACK_VERSION]:
    raise ValueError(f"The ATT&CK files are version {', '.join(versions)}, but ATTACK_VERSION is {ATTACK_VERSION}.")

meta = {
    'attack_version': ATTACK_VERSION,
    'generated_by': 'tools/build_attack_catalog.ipynb',
    'sources': {Path(path).name: file_hash(path) for path in [*source_files, dettect_file_path]},
}
meta

In [ ]:
# Build the catalog for all ATT&CK domains.
enterprise = process_mitre_attack_json(mitre_enterprise_file_path)
ics = process_mitre_attack_json(mitre_ics_file_path)
mobile = process_mitre_attack_json(mitre_mobile_file_path)

final_json = {
    "meta": meta,
    "enterprise": enterprise,
    "ics": ics,
    "mobile": mobile,
}

In [ ]:
# Check that every technique and sub-technique from the ATT&CK source files ended up in the catalog.
def check_all_techniques_present(catalog, domain_key, mitre_file_path):
    """Raise an error when (sub-)techniques from an ATT&CK source file are missing in the catalog."""
    mitre_attack_data = MitreAttackData(mitre_file_path)
    expected_ids = {x['external_references'][0]['external_id'] for x in mitre_attack_data.get_techniques(remove_revoked_deprecated=True)}

    generated_ids = set()
    for tactic in catalog[domain_key]['tactics']:
        for technique in tactic['techniques']:
            generated_ids.add(technique['external_id'])
            generated_ids.update(x['external_id'] for x in technique.get('sub_techniques', []))

    missing_ids = sorted(expected_ids - generated_ids)
    if missing_ids:
        raise ValueError(f"{domain_key}: {len(missing_ids)} (sub-)techniques are missing from the catalog: {missing_ids}")
    print(f"{domain_key}: all {len(expected_ids)} techniques and sub-techniques are present.")


# Check the occurrences: a parent technique counts every group and software that uses one of its sub-techniques,
# and every occurrence rank is between 0 and 1.
def check_occurrences(catalog, domain_key):
    """Raise an error when occurrence counts or ranks in the catalog are inconsistent."""
    for tactic in catalog[domain_key]['tactics']:
        for technique in tactic['techniques']:
            for sub_technique in technique.get('sub_techniques', []):
                for key in ['groups', 'software']:
                    missing = set(sub_technique[key]) - set(technique[key])
                    if missing:
                        raise ValueError(f"{domain_key}: {technique['external_id']} does not count the {key} of {sub_technique['external_id']}: {sorted(missing)}")
            for key in ['occurrence_total', 'occurrence_groups', 'occurrence_software']:
                rank = technique[f'{key}_order_normalized']
                if not 0 <= rank <= 1:
                    raise ValueError(f"{domain_key}: {technique['external_id']} has {key}_order_normalized {rank}, outside 0-1")
    print(f"{domain_key}: occurrence counts and ranks are consistent.")


for domain_key, mitre_file_path in [("enterprise", mitre_enterprise_file_path), ("ics", mitre_ics_file_path), ("mobile", mitre_mobile_file_path)]:
    check_all_techniques_present(final_json, domain_key, mitre_file_path)
    check_occurrences(final_json, domain_key)

In [ ]:
# Write the catalog for the app, via a temporary file in the same folder, so a stopped run never leaves half a file.
with tempfile.NamedTemporaryFile('w', dir=os.path.dirname(catalog_file_path), prefix='.catalog.', delete=False) as temporary:
    json.dump(final_json, temporary, indent=2)
os.replace(temporary.name, catalog_file_path)

print("JSON file created successfully!")

## Explore the source data

Starting points for looking at MITRE's data directly, for example when adapting to a new ATT&CK version.

In [ ]:
# The tactics (per matrix) and the techniques of the enterprise domain.
mitre_attack_data = MitreAttackData(mitre_enterprise_file_path)
techniques = mitre_attack_data.get_techniques(remove_revoked_deprecated=True)
list(mitre_attack_data.get_tactics_by_matrix().keys()), len(techniques)

In [ ]:
# A technique, with the data sources and data components that detect it.
technique = techniques[0]
technique.name, get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique.id)

In [ ]:
# A data component, with (the first ten of) the techniques it detects.
datacomponent = mitre_attack_data.get_datacomponents(remove_revoked_deprecated=True)[0]
datacomponent.name, [x['object'].name for x in mitre_attack_data.get_techniques_detected_by_datacomponent(datacomponent.id)][:10]